<a href="https://colab.research.google.com/github/vinifrts/padroes-de-corte/blob/main/padroes_de_corte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# importando tudo que nois precisa

!pip install ortools -q
from ortools.linear_solver import pywraplp
from google.colab import files


In [16]:
# abrindo o botao de upload
arquivo_enviado = files.upload()

# pegando o nome do arquivo enviado
nome_arquivo = list(arquivo_enviado.keys())[0]

# lendo o conteudo do arquivo e separando todos os numeros
with open(nome_arquivo, 'r') as arquivo:
    numeros = arquivo.read().split()

# tentando converter tudo p inteiro
try:
    numeros = [int(numero) for numero in numeros]
except ValueError:
    raise SystemExit("erro: o arquivo so pode ter numeros inteiros")

# conferindo se tem pelo menos os dois primeiros numeros
if len(numeros) < 2:
    raise SystemExit("erro: o arquivo precisa ter pelo menos o tamanho da barra e a quantidade de tipos")

# lendo o tamanho da barra
tamanho_barra = numeros[0]
if tamanho_barra <= 0:
    raise SystemExit("erro: o tamanho da barra tem que ser maior que zero")

# lendo quantos tipos de item tem na demanda
quantidade_tipos = numeros[1]
if quantidade_tipos < 1:
    raise SystemExit("erro: tem que ter pelo menos 1 tipo de item")

# conferindo se o arquivo tem a quantidade certa de numeros
if len(numeros) != 2 + 2 * quantidade_tipos:
    raise SystemExit("erro: a quantidade de numeros no arquivo nao bate com a quantidade de tipos")

# listas para guardar o tamanho e a demanda de cada tipo
tamanhos = []
demandas = []

# pegando os dados de cada tipo
for i in range(quantidade_tipos):
    tamanho_item = numeros[2 + 2 * i]
    demanda_item = numeros[3 + 2 * i]

    # conferindo o tamanho do item
    if tamanho_item <= 0:
        raise SystemExit("erro: o tamanho do item " + str(i + 1) + " tem que ser maior que zero")

    if tamanho_item > tamanho_barra:
        raise SystemExit("erro: o item " + str(i + 1) + " e maior que a barra, nao da para cortar")

    # conferindo a demanda do item
    if demanda_item < 0:
        raise SystemExit("erro: a demanda do item " + str(i + 1) + " nao pode ser negativa")

    tamanhos.append(tamanho_item)
    demandas.append(demanda_item)

# mostrando oq foi lido
print("tamanho da barra:", tamanho_barra)
print("quantidade de tipos de item:", quantidade_tipos)
print("tamanhos:", tamanhos)
print("demandas:", demandas)


Saving T2entrada-teste.txt to T2entrada-teste (2).txt
tamanho da barra: 150
quantidade de tipos de item: 3
tamanhos: [80, 60, 50]
demandas: [70, 100, 120]


In [17]:
# lista q vai guardar todos os padroes maximais
padroes = []

# funcao recursiva que testa todas as quantidades de cada item
def gerar_padroes(indice, padrao_atual, espaco_restante):

    # quando chega no ultimo item ai é hora de validar o padrao
    if indice == quantidade_tipos:

        # so vale se n for vazio
        if sum(padrao_atual) == 0:
            return

        # so vale se for maximal
        for tamanho in tamanhos:
            if tamanho <= espaco_restante:
                return

        padroes.append(list(padrao_atual))
        return

    # max de pecas desse item que cabem no espaco que sobrou
    maximo_pecas = espaco_restante // tamanhos[indice]

    # testando de zero ate o maximo
    for quantidade in range(maximo_pecas + 1):
        padrao_atual.append(quantidade)

        gerar_padroes(
            indice + 1,
            padrao_atual,
            espaco_restante - quantidade * tamanhos[indice]
        )

        padrao_atual.pop()

# chamando a funcao
gerar_padroes(0, [], tamanho_barra)


In [18]:
# calculando a sobra de cada padrao
sobras = []

for padrao in padroes:
    usado = 0

    for i in range(quantidade_tipos):
        usado = usado + padrao[i] * tamanhos[i]

    sobras.append(tamanho_barra - usado)

# mostrando os padroes
print("quantidade de padroes:", len(padroes))

for j in range(len(padroes)):
    print("padrao", j + 1, ":", padroes[j], "-> sobra:", sobras[j])


quantidade de padroes: 5
padrao 1 : [0, 0, 3] -> sobra: 0
padrao 2 : [0, 1, 1] -> sobra: 40
padrao 3 : [0, 2, 0] -> sobra: 30
padrao 4 : [1, 0, 1] -> sobra: 20
padrao 5 : [1, 1, 0] -> sobra: 10


In [19]:
# criando o solve
slv = pywraplp.Solver.CreateSolver('SCIP')

# variaveis x: quantas barras serao cortadas em cada padrao
x = []

for j in range(len(padroes)):
    x.append(slv.IntVar(0, slv.infinity(), 'x' + str(j + 1)))

# variaveis e: excedente produzido de cada item
excedentes = []

for i in range(quantidade_tipos):
    excedentes.append(
        slv.IntVar(0, slv.infinity(), 'e' + str(i + 1))
    )


In [20]:
# restricoes: producao - excedente = demanda
for i in range(quantidade_tipos):
    soma_producao = 0

    for j in range(len(padroes)):
        soma_producao = soma_producao + padroes[j][i] * x[j]

    slv.Add(soma_producao - excedentes[i] == demandas[i])

# funcao objetivo: minimizar o desperdicio total
desperdicio_total = 0

for j in range(len(padroes)):
    desperdicio_total = desperdicio_total + sobras[j] * x[j]

for i in range(quantidade_tipos):
    desperdicio_total = desperdicio_total + tamanhos[i] * excedentes[i]

slv.Minimize(desperdicio_total)


In [21]:
# resolvendo
status = slv.Solve()

# mostrando o modelo de PLI
print("=== modelo de pli ===")
print(slv.ExportModelAsLpFormat(False).lower())

=== modelo de pli ===
\ generated by mpmodelprotoexporter
\   name             : 
\   format           : free
\   constraints      : 3
\   variables        : 8
\     binary         : 0
\     integer        : 8
\     continuous     : 0
minimize
 obj: +40 x2 +30 x3 +20 x4 +10 x5 +80 e1 +60 e2 +50 e3 
subject to
 auto_c_000000000: +1 x4 +1 x5 -1 e1  = 70
 auto_c_000000001: +1 x2 +2 x3 +1 x5 -1 e2  = 100
 auto_c_000000002: +3 x1 +1 x2 +1 x4 -1 e3  = 120
bounds
 0 <= x1 <= inf
 0 <= x2 <= inf
 0 <= x3 <= inf
 0 <= x4 <= inf
 0 <= x5 <= inf
 0 <= e1 <= inf
 0 <= e2 <= inf
 0 <= e3 <= inf
generals
 x1
 x2
 x3
 x4
 x5
 e1
 e2
 e3
end



In [22]:
# tratando o resultado
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:

    if status == pywraplp.Solver.OPTIMAL:
        print("=== solucao otima ===")
    else:
        print("=== solucao viavel (n deu para provar que eh otima :( )) ===")

    # mostrando qtas barras usar de cada padrao
    total_barras = 0

    for j in range(len(padroes)):
        quantidade_barras = int(round(x[j].solution_value()))
        total_barras = total_barras + quantidade_barras

        print(
            "padrao", j + 1, padroes[j],
            "-> cortar", quantidade_barras, "barras"
        )

    # mostrando o excedente de cada item
    for i in range(quantidade_tipos):
        quantidade_excedente = int(round(excedentes[i].solution_value()))

        print(
            "excedente do item", i + 1, ":",
            quantidade_excedente, "pecas"
        )

    print("total de barras usadas:", total_barras)
    print("desperdicio associado:", slv.Objective().Value())

    # mostrando o total de cortes de cada tamanho
    print("=== total de cortes por tamanho ===")

    for i in range(quantidade_tipos):
        total_cortes = 0

        for j in range(len(padroes)):
            quantidade_barras = int(round(x[j].solution_value()))

            total_cortes = (
                total_cortes + padroes[j][i] * quantidade_barras
            )

        print(total_cortes, "cortes de", tamanhos[i], "metros")

elif status == pywraplp.Solver.INFEASIBLE:
    print("o modelo e inviavel, nao existe solucao")

else:
    print("nao achou solucao otima")


=== solucao otima ===
padrao 1 [0, 0, 3] -> cortar 40 barras
padrao 2 [0, 1, 1] -> cortar 0 barras
padrao 3 [0, 2, 0] -> cortar 15 barras
padrao 4 [1, 0, 1] -> cortar 0 barras
padrao 5 [1, 1, 0] -> cortar 70 barras
excedente do item 1 : 0 pecas
excedente do item 2 : 0 pecas
excedente do item 3 : 0 pecas
total de barras usadas: 125
desperdicio associado: 1150.0000000000007
=== total de cortes por tamanho ===
70 cortes de 80 metros
100 cortes de 60 metros
120 cortes de 50 metros
